# Notebook 02 — Data Preprocessing & Feature Engineering
## Du dataset brut au feature store pret-a-clusterer

---

## Section 0 — Contexte & Objectif du Pipeline

Le notebook EDA (`01_eda.ipynb`) a conclu avec 6 hypotheses de segmentation et une liste de 10 features a construire. Ce notebook transforme ces specifications en un feature store exploitable.

### Les 10 features cibles

| # | Feature | Source | Construction |
|---|---------|--------|--------------|
| 1 | `Log_Recency` | `last_purchase_date` | `log1p((max_date - last_date).days)` |
| 2 | `Log_Monetary` | `total_spent` | `log1p(total_spent)` apres clip P99 |
| 3 | `Frequency_flag` | `total_orders` | `int(total_orders >= 2)` |
| 4 | `avg_freight_ratio` | df_master (item level) | `mean(freight_value / price)` par client |
| 5 | `avg_delivery_delay` | df_master (order level) | `mean(actual - estimated)` par client |
| 6 | `avg_review_score` | `get_customer_aggregation()` | deja disponible |
| 7 | `payment_type_cc_flag` | `olist_order_payments` | `int(mode(payment_type) == 'credit_card')` |
| 8 | `avg_installments` | `olist_order_payments` | `mean(payment_installments)` par client |
| 9 | `region_freight_score` | `customer_state` | encodage ordinal region x charge de fret |
| 10 | `category_tier_encoded` | `product_category_name_english` | macro-segment dominant par client (ordinal 1-10) |

### Pipeline de transformation

```
PostgreSQL
  get_merged_dataframe()     -> df_master  (grain: order x item x payment)
  get_customer_aggregation() -> df_agg     (grain: customer, 6 cols)
  olist_order_payments       -> pay_raw    (grain: order x payment)
        |
  [Section 2] Feature Construction  (logistics + payments + geography + category)
        |
  [Section 3] Feature Store Assembly (merge sequentiel, grain: customer)
        |
  [Section 4] Outlier Treatment      (clip P99, bornes physiques)
        |
  [Section 5] Imputation             (mediane/zero, assert nulls == 0)
        |
  [Section 6] Log-Transform          (log1p Recency + Monetary)
        |
  [Section 7] Audit correlations & skewness (pre-scaling)
        |
  [Section 8] Export + Diagnostics   (parquet + PCA 2D sanity check)
```

> **Note clustering :** K-Means et CAH reposent sur des distances euclidiennes. Le `StandardScaler` sera applique dans le pipeline sklearn de chaque modele dans le notebook 03 — pas ici — pour eviter tout data leakage lors du scoring de nouveaux clients.

## Section 1 — Setup & Chargement des Sources

**Pourquoi 3 sources distinctes ?**
- `get_customer_aggregation()` donne le RFM de base (6 colonnes) mais pas les features comportementales
- `get_merged_dataframe()` est au grain `order x item x payment` — il faut dedupliquer avant d'agreger au niveau client
- `olist_order_payments` est charge separement pour construire proprement les features paiement (eviter les produits cartes du JOIN multi-table)

Regle : **toute agregation au niveau client doit passer par une deduplication explicite** au grain approprie (item, order, ou payment).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.append(str(PROJECT_ROOT / 'src'))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR    = PROJECT_ROOT / 'models'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

from data_loader import get_db_engine, get_merged_dataframe, get_customer_aggregation

engine = get_db_engine()

print('Chargement df_master (order x item x payment)...')
df_master = get_merged_dataframe(engine)
print('Chargement df_agg (customer level)...')
df_agg = get_customer_aggregation(engine)
print('Chargement olist_order_payments...')
pay_raw = pd.read_sql_table('olist_order_payments', engine)

STATE_REGION = {
    'AC': 'Norte',  'AM': 'Norte',  'AP': 'Norte',  'PA': 'Norte',
    'RO': 'Norte',  'RR': 'Norte',  'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste',
    'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste',
    'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste',
    'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul',
}
REGION_FREIGHT_ORDER = {'Sul': 1, 'Sudeste': 2, 'Centro-Oeste': 3, 'Nordeste': 4, 'Norte': 5}

print(f'df_master : {df_master.shape[0]:,} lignes x {df_master.shape[1]} colonnes')
print(f'df_agg    : {df_agg.shape[0]:,} clients')
print(f'pay_raw   : {pay_raw.shape[0]:,} lignes')

## Section 2 — Construction des Features Comportementales

### Regle de deduplication

Le `df_master` resulte d'un JOIN multi-table : pour chaque commande, les lignes d'items (`order_item_id`) sont croisees avec les lignes de paiement (`payment_sequential`). Une commande avec 2 articles et 2 modes de paiement genere **4 lignes** dans df_master.

Grain cible par feature :
- **Features item-level** (`freight_ratio`) — dedupliquer a `(order_id, order_item_id)` avant mean
- **Features order-level** (`delivery_delay_days`) — dedupliquer a `order_id` avant mean
- **Features paiement** — travailler depuis `pay_raw` joint a `olist_orders`

In [ ]:
# ── Pre-calcul des colonnes temporelles ──────────────────────────────────────
for col in ['order_purchase_timestamp', 'order_approved_at',
            'order_delivered_carrier_date', 'order_delivered_customer_date',
            'order_estimated_delivery_date']:
    df_master[col] = pd.to_datetime(df_master[col], errors='coerce')

df_master['actual_lead_time_days']    = (
    df_master['order_delivered_customer_date'] - df_master['order_purchase_timestamp']
).dt.total_seconds() / 86400
df_master['estimated_lead_time_days'] = (
    df_master['order_estimated_delivery_date'] - df_master['order_purchase_timestamp']
).dt.total_seconds() / 86400
df_master['delivery_delay_days'] = (
    df_master['actual_lead_time_days'] - df_master['estimated_lead_time_days']
)
df_master['freight_ratio'] = df_master['freight_value'] / df_master['price']
df_master['is_late']       = (df_master['delivery_delay_days'] > 0).astype(int)

# ── 2-A : Features logistiques ────────────────────────────────────────────────
df_items  = df_master.drop_duplicates(subset=['order_id', 'order_item_id']).copy()
df_orders = df_master.drop_duplicates(subset=['order_id']).copy()

customer_freight = df_items.groupby('customer_unique_id').agg(
    avg_freight_ratio=('freight_ratio', 'mean'),
).reset_index()

customer_logistics = df_orders.groupby('customer_unique_id').agg(
    avg_delivery_delay=('delivery_delay_days', 'mean'),
    avg_lead_time     =('actual_lead_time_days', 'mean'),
    pct_late_orders   =('is_late', 'mean'),
).reset_index()
customer_logistics = customer_logistics.merge(customer_freight, on='customer_unique_id', how='left')

print(f'Features logistiques : {len(customer_logistics):,} clients')
print(customer_logistics[['avg_freight_ratio', 'avg_delivery_delay']].describe().round(3))

### 2-B — Features paiement

Le mode de paiement dominant est determine par la valeur totale depensee avec chaque type (pas le count) : un client qui paie 80% en CB et 20% en boleto est classe CB.

In [ ]:
# ── 2-B : Features paiement ──────────────────────────────────────────────────
orders_map   = df_master[['order_id', 'customer_unique_id']].drop_duplicates('order_id')
pay_enriched = pay_raw.merge(orders_map, on='order_id', how='inner')

dominant_pay = (
    pay_enriched.groupby(['customer_unique_id', 'payment_type'])['payment_value']
    .sum().reset_index()
    .sort_values('payment_value', ascending=False)
    .drop_duplicates('customer_unique_id')[['customer_unique_id', 'payment_type']]
    .rename(columns={'payment_type': 'dominant_payment_type'})
)

avg_install = (
    pay_enriched.groupby('customer_unique_id')['payment_installments']
    .mean().reset_index()
    .rename(columns={'payment_installments': 'avg_installments'})
)

customer_payments = dominant_pay.merge(avg_install, on='customer_unique_id', how='left')
customer_payments['payment_type_cc_flag'] = (
    customer_payments['dominant_payment_type'] == 'credit_card'
).astype(int)

print(f'Features paiement : {len(customer_payments):,} clients')
print(customer_payments['payment_type_cc_flag']
      .value_counts(normalize=True).rename({0: 'Non-CB', 1: 'CB'}).round(3))

### 2-C — Feature geographique

L'etat modal par client est encode en score de fret ordinal : Sul (fret faible) = 1 a Norte (fret eleve) = 5. Ce score capture la correlation observee en EDA entre la region geographique et la charge de fret supportee par le client.

In [ ]:
# ── 2-C : Feature geographique ───────────────────────────────────────────────
customer_geo = (
    df_orders.groupby('customer_unique_id')['customer_state']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
)
customer_geo['customer_region']      = customer_geo['customer_state'].map(STATE_REGION)
customer_geo['region_freight_score'] = customer_geo['customer_region'].map(REGION_FREIGHT_ORDER)

unmapped = customer_geo['customer_region'].isna().sum()
print(f'Etats non mappes dans STATE_REGION : {unmapped} (attendu : 0)')
print('\nDistribution region_freight_score :')
print(customer_geo['region_freight_score'].value_counts().sort_index())

### 2-D — Construction de la Variable `category_tier` (Type de Produit)

#### Pourquoi cette variable ?

La colonne `product_category_name_english` comporte plus de 70 modalites brutes, ce qui la rend inexploitable directement en clustering. On regroupe ces categories en **10 macro-segments** coherents et interpretables (`category_tier`).

Ce regroupement capture le **type de besoin principal du client** tout en reduisant la dimensionnalite.

#### Agregation au niveau client

- **Client multi-achat** — on retient la categorie dominante (la plus frequente dans son historique d'achats)
- **Client mono-achat** — la categorie est directement celle du produit achete

#### Lien avec H5 (EDA)

L'hypothese H5 postule que la tolerance au delai varie selon la categorie d'achat. La validation ci-dessous le confirme ou l'infirme : si le review score moyen et le delai median varient de moins de 0.2 point entre les tiers, la feature apporte peu et pourra etre retiree du feature set.

In [ ]:
# ── Definition du mapping categorie -> macro-segment ─────────────────────────
CATEGORY_TIER_MAP = {
    # Maison & Ameublement
    'furniture_decor': 'Maison & Ameublement',
    'furniture_living_room': 'Maison & Ameublement',
    'furniture_bedroom': 'Maison & Ameublement',
    'furniture_mattress_and_upholstery': 'Maison & Ameublement',
    'housewares': 'Maison & Ameublement',
    'home_appliances': 'Maison & Ameublement',
    'home_appliances_2': 'Maison & Ameublement',
    'bed_bath_table': 'Maison & Ameublement',
    'kitchen_dining_laundry_garden_furniture': 'Maison & Ameublement',
    'home_comfort': 'Maison & Ameublement',
    'home_comfort_2': 'Maison & Ameublement',
    'small_appliances': 'Maison & Ameublement',
    'small_appliances_home_oven_and_coffee': 'Maison & Ameublement',
    'air_conditioning': 'Maison & Ameublement',
    # Electronique & Technologie
    'electronics': 'Electronique & Technologie',
    'computers_accessories': 'Electronique & Technologie',
    'computers': 'Electronique & Technologie',
    'telephony': 'Electronique & Technologie',
    'audio': 'Electronique & Technologie',
    'tablets_printing_image': 'Electronique & Technologie',
    'signaling_and_security': 'Electronique & Technologie',
    'fixed_telephony': 'Electronique & Technologie',
    'portable_kitchen_food_processors': 'Electronique & Technologie',
    # Beaute & Sante
    'health_beauty': 'Beaute & Sante',
    'perfumery': 'Beaute & Sante',
    'diapers_and_hygiene': 'Beaute & Sante',
    # Mode & Accessoires
    'fashion_bags_accessories': 'Mode & Accessoires',
    'fashion_female_clothing': 'Mode & Accessoires',
    'fashion_male_clothing': 'Mode & Accessoires',
    'fashion_shoes': 'Mode & Accessoires',
    'fashion_sport': 'Mode & Accessoires',
    'fashion_underwear_beach': 'Mode & Accessoires',
    'fashion_childrens_clothes': 'Mode & Accessoires',
    'watches_gifts': 'Mode & Accessoires',
    'luggage_accessories': 'Mode & Accessoires',
    'costumes_accessories': 'Mode & Accessoires',
    # Loisirs & Culture
    'books_general_interest': 'Loisirs & Culture',
    'books_technical': 'Loisirs & Culture',
    'books_imported': 'Loisirs & Culture',
    'music': 'Loisirs & Culture',
    'dvds_blu_ray': 'Loisirs & Culture',
    'art': 'Loisirs & Culture',
    'stationery': 'Loisirs & Culture',
    'cds_dvds_musicals': 'Loisirs & Culture',
    'musical_instruments': 'Loisirs & Culture',
    'la_cuisine': 'Loisirs & Culture',
    # Bricolage & Construction
    'construction_tools_construction': 'Bricolage & Construction',
    'construction_tools_tools': 'Bricolage & Construction',
    'construction_tools_safety': 'Bricolage & Construction',
    'construction_tools_lights': 'Bricolage & Construction',
    'garden_tools': 'Bricolage & Construction',
    'flowers': 'Bricolage & Construction',
    # Enfants & Loisirs
    'toys': 'Enfants & Loisirs',
    'baby': 'Enfants & Loisirs',
    'sports_leisure': 'Enfants & Loisirs',
    'christmas_supplies': 'Enfants & Loisirs',
    # Automobile & Industriel
    'auto': 'Automobile & Industriel',
    'industry_commerce_and_business': 'Automobile & Industriel',
    'office_furniture': 'Automobile & Industriel',
    'market_place': 'Automobile & Industriel',
    # Animaux
    'pet_shop': 'Animaux',
    'food': 'Animaux',
    'food_drink': 'Animaux',
    # Divers (fallback)
    'cool_stuff': 'Divers',
    'party_supplies': 'Divers',
    'agro_industry_and_commerce': 'Divers',
    'consoles_games': 'Divers',
    'security_and_services': 'Divers',
}

TIER_ORDER = [
    'Maison & Ameublement', 'Electronique & Technologie', 'Beaute & Sante',
    'Mode & Accessoires', 'Loisirs & Culture', 'Bricolage & Construction',
    'Enfants & Loisirs', 'Automobile & Industriel', 'Animaux', 'Divers',
]
TIER_ENCODING = {tier: i + 1 for i, tier in enumerate(TIER_ORDER)}

print(f'{len(CATEGORY_TIER_MAP)} categories mappees vers {len(TIER_ORDER)} macro-segments')

On applique le mapping sur `df_items` (grain order x item), puis on extrait la categorie dominante par client. Les categories non mappees tombent dans le tier `Divers`.

In [ ]:
# ── Application du mapping + agregation par client ───────────────────────────
df_items['category_tier'] = (
    df_items['product_category_name_english'].map(CATEGORY_TIER_MAP).fillna('Divers')
)

unmapped_cats = (
    df_items[df_items['category_tier'] == 'Divers']['product_category_name_english']
    .value_counts().head(10)
)
print("Categories tombant dans 'Divers' (top 10) :")
print(unmapped_cats.to_string())

customer_category = (
    df_items.groupby(['customer_unique_id', 'category_tier'])
    .size().reset_index(name='n_items')
    .sort_values('n_items', ascending=False)
    .drop_duplicates('customer_unique_id')
    [['customer_unique_id', 'category_tier']]
)
customer_category['category_tier_encoded'] = customer_category['category_tier'].map(TIER_ENCODING)

tier_dist = (
    customer_category['category_tier'].value_counts()
    .reindex(TIER_ORDER).reset_index()
)
tier_dist.columns = ['category_tier', 'n_clients']
tier_dist['pct'] = (tier_dist['n_clients'] / len(customer_category) * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 5))
tier_plot = tier_dist.dropna(subset=['n_clients']).sort_values('n_clients', ascending=True)
bars = ax.barh(tier_plot['category_tier'], tier_plot['n_clients'],
               color=plt.cm.tab10.colors[:len(tier_plot)], edgecolor='white', linewidth=1.2)
for bar, pct in zip(bars, tier_plot['pct']):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%', va='center', fontsize=9, color='gray')
ax.set_xlabel("Nombre de clients (categorie dominante)")
ax.set_title("Distribution des Macro-Segments Produit (category_tier)", fontsize=12)
plt.tight_layout()
plt.show()

#### Validation H5 — Review score et delai par macro-segment

Si la variation inter-tiers est inferieure a 0.2 point sur le review score et inferieure a 2 jours sur le delai, `category_tier_encoded` apporte peu et peut etre retiree du feature set final.

In [ ]:
# ── Validation H5 : review score et delai median par tier ────────────────────
tier_review = (
    df_master[['order_id', 'product_category_name_english',
               'review_score', 'delivery_delay_days']]
    .drop_duplicates(subset=['order_id', 'product_category_name_english'])
    .assign(category_tier=lambda x:
            x['product_category_name_english'].map(CATEGORY_TIER_MAP).fillna('Divers'))
    .groupby('category_tier')
    .agg(
        review_score_mean=('review_score',       'mean'),
        delay_median     =('delivery_delay_days', 'median'),
        n_orders         =('order_id',            'nunique'),
    )
    .reindex(TIER_ORDER).round(3)
)

display(tier_review.style
    .format({'review_score_mean': '{:.3f}', 'delay_median': '{:.1f}', 'n_orders': '{:,}'})
    .background_gradient(cmap='RdYlGn',   subset=['review_score_mean'])
    .background_gradient(cmap='RdYlGn_r', subset=['delay_median'])
)

score_range = tier_review['review_score_mean'].max() - tier_review['review_score_mean'].min()
delay_range = tier_review['delay_median'].max()       - tier_review['delay_median'].min()
print(f'Variation inter-tiers review score : {score_range:.3f} (seuil : 0.20)')
print(f'Variation inter-tiers delai median : {delay_range:.1f} j (seuil : 2.0 j)')
h5_confirmed = score_range >= 0.20 or delay_range >= 2.0
print(f'H5 {"confirmee" if h5_confirmed else "non confirmee"} — category_tier_encoded '
      f'{"conservee" if h5_confirmed else "a retirer du feature set"}')

## Section 3 — Assemblage du Feature Store Client

On fusionne les 4 blocs de features autour du `customer_unique_id` par left join depuis `df_agg`. Le left join garantit que tous les clients avec au moins une commande livree sont representes, meme si leur etat geographique ou type de paiement est manquant (ces cas seront imputes en Section 5).

On calcule ensuite les metriques RFM de base : Recency, Frequency, Monetary.

In [ ]:
# ── Merge sequentiel autour de customer_unique_id ─────────────────────────────
df_features = df_agg.copy()

merge_steps = [
    (customer_logistics[['customer_unique_id', 'avg_freight_ratio',
                          'avg_delivery_delay', 'avg_lead_time',
                          'pct_late_orders']], 'Logistique'),
    (customer_payments[['customer_unique_id', 'avg_installments',
                         'payment_type_cc_flag',
                         'dominant_payment_type']], 'Paiements'),
    (customer_geo[['customer_unique_id', 'customer_state',
                   'customer_region', 'region_freight_score']], 'Geographie'),
    (customer_category[['customer_unique_id', 'category_tier',
                         'category_tier_encoded']], 'Category Tier'),
]

for df_extra, label in merge_steps:
    before = len(df_features)
    df_features = df_features.merge(df_extra, on='customer_unique_id', how='left')
    assert len(df_features) == before, f'Merge {label} a cree des doublons !'
    print(f'Merge {label:<15} : {len(df_features):,} clients, {df_features.shape[1]} colonnes')

max_date = pd.to_datetime(df_features['last_purchase_date']).max()
df_features['Recency']        = (max_date - pd.to_datetime(df_features['last_purchase_date'])).dt.days
df_features['Frequency']      = df_features['total_orders']
df_features['Monetary']       = df_features['total_spent']
df_features['Frequency_flag'] = (df_features['Frequency'] >= 2).astype(int)

print(f'\nDate de reference : {max_date.date()}')
print(f'Feature store brut : {df_features.shape[0]:,} clients x {df_features.shape[1]} colonnes')

FINAL_FEATURES = [
    'Log_Recency', 'Log_Monetary', 'Frequency_flag',
    'avg_freight_ratio', 'avg_delivery_delay', 'avg_review_score',
    'payment_type_cc_flag', 'avg_installments', 'region_freight_score',
    'category_tier_encoded',
]

key_cols = [f.replace('Log_', '') if f.startswith('Log_') else f for f in FINAL_FEATURES]
key_cols = list(dict.fromkeys(
    ['Recency', 'Monetary', 'Frequency_flag', 'avg_freight_ratio',
     'avg_delivery_delay', 'avg_review_score', 'payment_type_cc_flag',
     'avg_installments', 'region_freight_score', 'category_tier_encoded']
))
null_audit = df_features[key_cols].isnull().mean() * 100
display(null_audit.rename('% nulls').to_frame().style
        .format('{:.2f}%').background_gradient(cmap='Reds', vmin=0, vmax=10))

## Section 4 — Traitement des Valeurs Aberrantes (Outliers)

**Strategie : clip conservateur, jamais suppression.** Les outliers sont des valeurs reelles — les supprimer serait une perte d'information. On les ramene aux bornes definies par l'EDA ou par des contraintes physiques.

| Feature | Borne | Justification |
|---------|-------|---------------|
| `Monetary` | P99 | Queue droite longue — preserve les outliers high-value |
| `avg_freight_ratio` | (0, 2.0) | Ratio > 2 = erreur de saisie ou prix = 0 |
| `avg_delivery_delay` | (-30, 60) | -30j = livraison tres en avance ; +60j = retard extreme |
| `avg_installments` | (1, 12) | Maximum legal carte de credit bresilien standard |

In [ ]:
# ── Clipping ──────────────────────────────────────────────────────────────────
outlier_cols = ['Monetary', 'avg_freight_ratio', 'avg_delivery_delay', 'avg_installments']
before_clip  = {c: df_features[c].copy() for c in outlier_cols}

p99_monetary = df_features['Monetary'].quantile(0.99)
df_features['Monetary']           = df_features['Monetary'].clip(upper=p99_monetary)
df_features['avg_freight_ratio']  = df_features['avg_freight_ratio'].clip(lower=0, upper=2.0)
df_features['avg_delivery_delay'] = df_features['avg_delivery_delay'].clip(lower=-30, upper=60)
df_features['avg_installments']   = df_features['avg_installments'].clip(lower=1, upper=12)

print(f'Monetary P99 (borne superieure) : R$ {p99_monetary:,.0f}')
print('\nValeurs affectees par le clipping :')
for col, before in before_clip.items():
    n = (before != df_features[col]).sum()
    print(f'  {col:<25} : {n:>5,} clients ({n / len(df_features) * 100:.2f}%)')

In [ ]:
# ── Visualisation avant / apres clipping ─────────────────────────────────────
titles = ['Monetary (BRL)', 'Freight Ratio', 'Delai Livraison (j)', 'Nb Echeances']
fig, axes = plt.subplots(2, 4, figsize=(18, 7))

for i, (col, title) in enumerate(zip(outlier_cols, titles)):
    sns.boxplot(y=before_clip[col].dropna(), ax=axes[0, i], color='#e74c3c', width=0.4)
    axes[0, i].set_title(f'{title}\nAvant clipping', fontsize=10)
    sns.boxplot(y=df_features[col].dropna(), ax=axes[1, i], color='#27ae60', width=0.4)
    axes[1, i].set_title(f'{title}\nApres clipping', fontsize=10)

plt.suptitle('Outlier Treatment — Avant vs Apres Clipping', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Section 5 — Imputation des Valeurs Manquantes

Apres le clipping, certaines valeurs nulles subsistent pour des raisons structurelles :
- `avg_freight_ratio` — commandes avec `price = 0` → ratio indefini → imputer mediane nationale
- `avg_delivery_delay` — toutes les commandes d'un client manquent de date → imputer 0
- `region_freight_score` — etats non mappes → imputer 3 (Centro-Oeste = mediane nationale)
- `avg_review_score` — clients sans review → imputer mediane nationale
- `avg_installments` — clients sans paiement CB → imputer 1 (pas d'echelonnement)

L'assertion finale est un garde-fou : si des nulls residuels persistent, le notebook s'arrete avec une erreur explicite.

In [ ]:
# ── Imputation ────────────────────────────────────────────────────────────────
IMPUTATION_MAP = {
    'avg_freight_ratio':    df_features['avg_freight_ratio'].median(),
    'avg_delivery_delay':   0.0,
    'avg_review_score':     df_features['avg_review_score'].median(),
    'payment_type_cc_flag': 0,
    'avg_installments':     1.0,
    'region_freight_score': 3,
    'category_tier_encoded': TIER_ENCODING['Divers'],
}

print('Valeurs imputees :')
for col, val in IMPUTATION_MAP.items():
    n_null = df_features[col].isna().sum()
    if n_null > 0:
        print(f'  {col:<25} : {n_null:>5,} nulls -> impute a {val}')

df_features.fillna(IMPUTATION_MAP, inplace=True)

check_cols = list(IMPUTATION_MAP.keys()) + ['Recency', 'Monetary']
residual = df_features[check_cols].isnull().sum()
residual = residual[residual > 0]
assert len(residual) == 0, f'Nulls residuels detectes : {residual.to_dict()}'
print('\nAssertion OK — aucun null residuel dans les features finales.')

## Section 6 — Log-Transformation & Constitution du Feature Set Final

La **log-transformation** (`log1p = log(x+1)`) resout deux problemes :
1. **Skewness** — `Recency` et `Monetary` ont des distributions fortement asymetriques (skewness > 3 confirme en EDA). K-Means est sensible au skewness — un client a Monetary = 5 000 BRL dominerait tous les clusters avec les valeurs brutes.
2. **Outliers residuels** — apres clipping, des valeurs extremes persistent. Le log les compresse vers le centre de la distribution.

`Frequency_flag` reste binaire (0/1) — une log-transformation n'apporterait rien.

In [ ]:
# ── Log-transformations ────────────────────────────────────────────────────────
df_features['Log_Recency']  = np.log1p(df_features['Recency'])
df_features['Log_Monetary'] = np.log1p(df_features['Monetary'])

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
pairs = [
    ('Recency',  'Log_Recency',  '#3498db', '#85c1e9'),
    ('Monetary', 'Log_Monetary', '#27ae60', '#7dcea0'),
]
for col_idx, (raw, log_col, cr, cl) in enumerate(pairs):
    skew_raw = df_features[raw].skew()
    skew_log = df_features[log_col].skew()
    sns.histplot(df_features[raw],     bins=60, kde=True, ax=axes[0, col_idx], color=cr)
    axes[0, col_idx].set_title(f'{raw} brut (skew = {skew_raw:.2f})')
    sns.histplot(df_features[log_col], bins=60, kde=True, ax=axes[1, col_idx], color=cl)
    axes[1, col_idx].set_title(f'log({raw}+1) (skew = {skew_log:.2f})')

plt.suptitle('Log-Transformation — Reduction du Skewness', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Resume du feature set final ───────────────────────────────────────────────
summary = pd.DataFrame({
    'Feature':  FINAL_FEATURES,
    'Mean':     df_features[FINAL_FEATURES].mean().round(3).values,
    'Std':      df_features[FINAL_FEATURES].std().round(3).values,
    'Skewness': df_features[FINAL_FEATURES].skew().round(3).values,
    'Min':      df_features[FINAL_FEATURES].min().round(3).values,
    'Max':      df_features[FINAL_FEATURES].max().round(3).values,
})
display(summary.style
    .format('{:.3f}', subset=['Mean', 'Std', 'Skewness', 'Min', 'Max'])
    .background_gradient(cmap='RdYlGn_r', subset=['Skewness'], vmin=-2, vmax=2))

## Section 7 — Audit Correlations & Skewness Pre-Export

Le scaling est intentionnellement **absent de ce notebook**. Chaque algorithme a des besoins differents :

| Algorithme | Scaling recommande |
|---|---|
| K-Means | `StandardScaler` |
| DBSCAN | `RobustScaler` (moins sensible aux outliers residuels) |
| CAH (Ward) | `StandardScaler` |
| Random Forest (interpretation) | Aucun scaling |

Le scaling sera applique **dans le pipeline sklearn de chaque modele** dans le notebook 03, ce qui garantit qu'il est encapsule et reproductible.

In [ ]:
# ── Heatmap de correlation (triangle inferieur) ──────────────────────────────
X_raw = df_features[FINAL_FEATURES].copy()
corr  = X_raw.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, mask=mask, ax=ax,
            linewidths=0.4, linecolor='white', annot_kws={'size': 9})
ax.set_title(f'Correlations entre les {len(FINAL_FEATURES)} features — Pre-scaling', fontsize=12)
plt.tight_layout()
plt.show()

high_corr = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack().reset_index()
    .rename(columns={'level_0': 'Feature A', 'level_1': 'Feature B', 0: 'Correlation'})
)
high_corr = high_corr[high_corr['Correlation'].abs() > 0.6].sort_values('Correlation', ascending=False)
if len(high_corr) > 0:
    print('Paires fortement correlees (|rho| > 0.6) — a surveiller :')
    display(high_corr.style.format({'Correlation': '{:.3f}'}).hide(axis='index'))
else:
    print('Aucune paire avec |rho| > 0.6 — features bien decorrelees.')

In [ ]:
# ── Audit skewness pre-scaling ────────────────────────────────────────────────
skew_df = X_raw.skew().round(3).reset_index()
skew_df.columns = ['Feature', 'Skewness']
skew_df['Statut'] = skew_df['Skewness'].apply(
    lambda x: 'Eleve (>2)'   if abs(x) > 2
    else      ('Modere (1-2)' if abs(x) > 1 else 'OK (<1)')
)

def _skew_color(v):
    if 'Eleve'  in str(v): return 'background-color: #f5b7b1'
    if 'Modere' in str(v): return 'background-color: #fef9e7'
    if 'OK'     in str(v): return 'background-color: #eafaf1'
    return ''

display(skew_df.style
    .map(_skew_color, subset=['Statut'])
    .format({'Skewness': '{:.3f}'})
    .hide(axis='index')
)
print('Les features Eleve seront compressee par le StandardScaler dans le pipeline notebook 03.')

## Section 8 — Export du Feature Store & Sanity Check PCA

Un seul artefact est exporte depuis ce notebook :
- **`customer_features_raw.parquet`** — feature store log-transforme mais non scale, input du notebook 03.

Le fichier contient deux versions de chaque feature :
- **Version brute interpretable** (`Recency`, `Monetary`, `category_tier`...) — pour l'interpretation des clusters
- **Version feature-engineered** (`Log_Recency`, `Log_Monetary`, `category_tier_encoded`...) — pour les algorithmes

La **projection PCA 2D** est un sanity check visuel : si une structure est visible meme sans normalisation, les features discriminent bien les clients.

In [ ]:
# ── Export Parquet ────────────────────────────────────────────────────────────
raw_path = PROCESSED_DIR / 'customer_features_raw.parquet'

COLS_EXPORT = [
    'customer_unique_id',
    'Recency', 'Monetary', 'Frequency', 'Frequency_flag',
    'avg_review_score', 'avg_delivery_delay', 'avg_lead_time',
    'avg_freight_ratio', 'avg_installments', 'pct_late_orders',
    'payment_type_cc_flag', 'dominant_payment_type',
    'customer_state', 'customer_region', 'region_freight_score',
    'category_tier', 'category_tier_encoded',
    'Log_Recency', 'Log_Monetary',
]
cols_export = [c for c in COLS_EXPORT if c in df_features.columns]
df_export   = df_features[cols_export].copy()
df_export.to_parquet(raw_path, index=False)

null_pct = df_features[FINAL_FEATURES].isnull().mean().mean() * 100
skew_max = df_features[FINAL_FEATURES].skew().abs().max()
freq_flag_pct = df_features['Frequency_flag'].mean() * 100

print(f'Feature store exporte : {raw_path}')
print(f'  Clients            : {len(df_export):,}')
print(f'  Colonnes exportees : {len(cols_export)}')
print(f'  Features clustering: {len(FINAL_FEATURES)}')
print(f'  Nulls residuels    : {null_pct:.2f}%')
print(f'  Skewness max       : {skew_max:.3f}')
print(f'  Frequency_flag=1   : {freq_flag_pct:.1f}%')
print(f'  Scaling            : non — delegueau pipeline notebook 03')

In [ ]:
# ── PCA 2D — Sanity Check Visuel ─────────────────────────────────────────────
X_pca_input = StandardScaler().fit_transform(X_raw)  # scaling temporaire pour PCA uniquement
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_pca_input)
var_explained = pca.explained_variance_ratio_.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                      c=df_features['Log_Monetary'].values,
                      cmap='viridis', alpha=0.15, s=3)
plt.colorbar(sc1, ax=axes[0], label='Log_Monetary')
axes[0].set_title(f'PCA 2D — colore par Log_Monetary\nVariance expliquee : {var_explained:.1f}%')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')

sc2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                      c=df_features['Log_Recency'].values,
                      cmap='plasma', alpha=0.15, s=3)
plt.colorbar(sc2, ax=axes[1], label='Log_Recency')
axes[1].set_title(f'PCA 2D — colore par Log_Recency\n(structure = features discriminantes)')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')

plt.suptitle('Sanity Check — Projection PCA 2D (scaling temporaire, non exporte)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Loadings PCA ─────────────────────────────────────────────────────────────
loadings = pd.DataFrame(
    pca.components_.T,
    index=FINAL_FEATURES,
    columns=['PC1', 'PC2']
).round(3)
display(loadings.style.background_gradient(cmap='RdBu', vmin=-1, vmax=1).format('{:.3f}'))
print(f'Variance expliquee PC1 : {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'Variance expliquee PC2 : {pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'Total PC1+PC2          : {var_explained:.1f}%')
print(f'-> Notebook 03 chargera {raw_path.name} et appliquera le scaling dans chaque pipeline.')